# TF-IDF word + char n-gram + Linear SVM

Notebook nay thu nghiem phuong phap TF-IDF ket hop word n-gram va character n-gram, sau do dung Linear SVM cho bai toan multi-label aspect detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, hamming_loss, multilabel_confusion_matrix

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

RANDOM_STATE = 42

## 1. Load data

In [ ]:
df = pd.read_csv('Data/Restaurant_ABSA_processed.csv')

aspect_cols = ['food', 'service', 'price', 'ambiance', 'miscellaneous']
text_col = 'review_cleaned'

df[text_col] = df[text_col].fillna('')
X = df[text_col]
y = df[aspect_cols].astype(int)

print(df.shape)
display(df.head())
display(y.sum().sort_values(ascending=False).rename('positive_count').to_frame())

## 2. Train / validation / test split

Validation set duoc dung de tune threshold cho diem `decision_function` cua Linear SVM. Test set chi dung de danh gia cuoi cung.

In [ ]:
def stratify_if_possible(labels):
    counts = pd.Series(labels).value_counts()
    if counts.min() >= 2:
        return labels
    return None


label_combo = y.astype(str).agg(''.join, axis=1)

X_train_val, X_test, y_train_val, y_test, combo_train_val, combo_test = train_test_split(
    X,
    y,
    label_combo,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_if_possible(label_combo),
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_if_possible(combo_train_val),
)

print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)

display(pd.DataFrame({
    'train': y_train.sum(),
    'validation': y_val.sum(),
    'test': y_test.sum(),
}))

## 3. Model: word TF-IDF + char TF-IDF + Linear SVM

In [ ]:
word_tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3),
    max_features=8000,
    min_df=1,
    sublinear_tf=True,
)

char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=12000,
    min_df=1,
    sublinear_tf=True,
)

model = Pipeline([
    ('features', FeatureUnion([
        ('word_tfidf', word_tfidf),
        ('char_tfidf', char_tfidf),
    ], n_jobs=-1)),
    ('clf', OneVsRestClassifier(
        LinearSVC(
            C=1.0,
            class_weight='balanced',
            max_iter=10000,
            random_state=RANDOM_STATE,
        )
    )),
])

model.fit(X_train, y_train)
model

## 4. Tune threshold tren validation set

Linear SVM khong tra ve probability truc tiep. Ta dung `decision_function`, sau do tim threshold tot nhat cho tung aspect tren validation set.

In [ ]:
def tune_thresholds(y_true, scores, aspect_cols, grid_size=101):
    y_true = np.asarray(y_true)
    thresholds = {}
    rows = []

    for i, aspect in enumerate(aspect_cols):
        aspect_scores = scores[:, i]
        candidates = np.linspace(aspect_scores.min(), aspect_scores.max(), grid_size)
        candidates = np.unique(np.concatenate([candidates, [0.0]]))

        best_threshold = 0.0
        best_f1 = -1.0

        for threshold in candidates:
            pred = (aspect_scores >= threshold).astype(int)
            f1 = f1_score(y_true[:, i], pred, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold

        thresholds[aspect] = best_threshold
        rows.append({
            'aspect': aspect,
            'threshold': best_threshold,
            'validation_f1': best_f1,
        })

    return thresholds, pd.DataFrame(rows)


def predict_with_thresholds(scores, thresholds, aspect_cols, top_k=None, ensure_one_label=True):
    pred = np.zeros(scores.shape, dtype=int)

    for i, aspect in enumerate(aspect_cols):
        pred[:, i] = (scores[:, i] >= thresholds[aspect]).astype(int)

    if ensure_one_label:
        empty_rows = np.where(pred.sum(axis=1) == 0)[0]
        if len(empty_rows) > 0:
            pred[empty_rows, np.argmax(scores[empty_rows], axis=1)] = 1

    if top_k is not None:
        for row_idx in range(pred.shape[0]):
            if pred[row_idx].sum() > top_k:
                keep = np.argsort(scores[row_idx])[-top_k:]
                pred[row_idx] = 0
                pred[row_idx, keep] = 1

    return pred


val_scores = model.decision_function(X_val)
thresholds, threshold_report = tune_thresholds(y_val.values, val_scores, aspect_cols)

display(threshold_report)

## 5. Danh gia tren validation set

In [ ]:
def evaluate_multilabel(y_true, y_pred, aspect_cols, title='Evaluation'):
    print(title)
    print('-' * len(title))
    print(classification_report(y_true, y_pred, target_names=aspect_cols, zero_division=0))
    print('F1 Micro:', f1_score(y_true, y_pred, average='micro', zero_division=0))
    print('F1 Macro:', f1_score(y_true, y_pred, average='macro', zero_division=0))
    print('F1 Samples:', f1_score(y_true, y_pred, average='samples', zero_division=0))
    print('Hamming Loss:', hamming_loss(y_true, y_pred))


y_val_pred_default = (val_scores >= 0.0).astype(int)
y_val_pred_tuned = predict_with_thresholds(val_scores, thresholds, aspect_cols)
y_val_pred_tuned_top2 = predict_with_thresholds(val_scores, thresholds, aspect_cols, top_k=2)

evaluate_multilabel(y_val.values, y_val_pred_default, aspect_cols, title='Validation - default threshold 0')
print('\n')
evaluate_multilabel(y_val.values, y_val_pred_tuned, aspect_cols, title='Validation - tuned thresholds')
print('\n')
evaluate_multilabel(y_val.values, y_val_pred_tuned_top2, aspect_cols, title='Validation - tuned thresholds + top_k=2')

## 6. Danh gia cuoi cung tren test set

In [ ]:
test_scores = model.decision_function(X_test)

y_test_pred_default = (test_scores >= 0.0).astype(int)
y_test_pred_tuned = predict_with_thresholds(test_scores, thresholds, aspect_cols)
y_test_pred_tuned_top2 = predict_with_thresholds(test_scores, thresholds, aspect_cols, top_k=2)

evaluate_multilabel(y_test.values, y_test_pred_default, aspect_cols, title='Test - default threshold 0')
print('\n')
evaluate_multilabel(y_test.values, y_test_pred_tuned, aspect_cols, title='Test - tuned thresholds')
print('\n')
evaluate_multilabel(y_test.values, y_test_pred_tuned_top2, aspect_cols, title='Test - tuned thresholds + top_k=2')

## 7. Confusion matrix tung aspect

In [ ]:
best_test_pred = y_test_pred_tuned_top2
mcm = multilabel_confusion_matrix(y_test.values, best_test_pred)

plt.figure(figsize=(9, 11))
for i, aspect in enumerate(aspect_cols):
    plt.subplot(3, 2, i + 1)
    sns.heatmap(
        mcm[i],
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Pred 0', 'Pred 1'],
        yticklabels=['True 0', 'True 1'],
    )
    plt.title(aspect)

plt.tight_layout()
plt.show()

## 8. Xem cac mau du doan sai

In [ ]:
results = pd.DataFrame({
    'review_cleaned': X_test.values,
    'true_aspects': [', '.join(np.array(aspect_cols)[row.astype(bool)]) for row in y_test.values],
    'pred_aspects': [', '.join(np.array(aspect_cols)[row.astype(bool)]) for row in best_test_pred],
})

misclassified = results[results['true_aspects'] != results['pred_aspects']]
print('Misclassified:', len(misclassified), '/', len(results))
display(misclassified.head(20))

## 9. Thu du doan review moi

In [ ]:
new_reviews = [
    'The food was tasty but the price was too high.',
    'Great ambiance and friendly service.',
    'The place was clean and the staff were quick.',
]

new_scores = model.decision_function(new_reviews)
new_pred = predict_with_thresholds(new_scores, thresholds, aspect_cols, top_k=2)

for review, pred_row in zip(new_reviews, new_pred):
    aspects = np.array(aspect_cols)[pred_row.astype(bool)]
    print(review)
    print('Predicted aspects:', ', '.join(aspects))
    print()